<center>
<img src="img/logo-hds.png" width="650"/>

***

<center>Lecture 10</center>

***

<center>Multimodal Health Data</center>

***

<center>16 November 2026</center>
<center>Rahman Peimankar</center>

## What can a waveform and a table tell us together?

By the end of today, you should be able to match two inputs to the same observation, explain early and late fusion, and check whether combining them helps.

**Today’s question:** Can an ECG window and patient age help identify a window with a higher monitor respiratory rate?


## Two inputs describe different things

<img src="img/ecg.png" width="780" alt="Illustrative ECG waveform with repeated beats">

| Input | What a model can use |
|:---|:---|
| ECG time series | A summary of how the signal varies within a window |
| Patient table | Age, which stays fixed across windows from the same patient |

**Discuss:** If we take two ECG windows from one patient, which input changes?

## One prediction refers to one patient and window

**Example**

| ECG patient | ECG window | Age-table patient | Age | Pair? |
|:---|:---|:---|:---|:---|
| A | 0–30 s | A | 68 | Yes |
| A | 30–60 s | A | 68 | Yes |
| B | 0–30 s | A | 68 | No |

Use the patient identifier to match age to ECG. Keep the recording and time interval to identify the window. Split by patient so their windows stay together.

## Early fusion combines features

<img src="img/early.svg" width="950" alt="ECG summaries and patient age join into one feature row for a single logistic model">

**Input row:** ECG spread 0.7, ECG change 0.2, age 68.

**Predict:** If we change age to 42 while keeping the ECG fixed, which feature changes? Could the joint model change its prediction?

## What early fusion keeps

| Advantages | Limitations |
|:---|:---|
| Keeps the individual features available to the model | Needs correctly matched inputs |
| Learns the feature weights together | Needs a plan for missing age or ECG |
| Fits one prediction model | Extra features can add noise |



## Late fusion combines predictions

<img src="img/late.svg" width="950" alt="Separate ECG and age models produce probabilities that are averaged">

**Example:** ECG model says 0.80, age model says 0.40. Their mean is 0.60, giving class 1 at a threshold of 0.50.

**Predict:** If both models give probabilities below 0.50, can their mean reach 0.50?

## What late fusion trades away

| Advantages | Limitations |
|:---|:---|
| Develop and inspect each model separately | The average only receives probabilities |
| Can plan a fallback to the available model | Equal weights ignore differences in model reliability |

**Pair A:** (0.90, 0.10) → 0.50. **Pair B:** (0.50, 0.50) → 0.50.

The averages are identical. Which pair shows stronger disagreement?

## How do we tell whether fusion helps?

Compare ECG only, age only, early fusion and late fusion on **the same held-out patients and windows**.

**For example:** On 20 windows, an ECG model catches 2 of 5 positives and raises one false alarm. A fused model catches 4 of 5 and raises three false alarms. Both make four mistakes.

**Discuss:** Why is accuracy identical? Which error matters more for the intended use?

## Summary

| Aspect | Early fusion | Late fusion |
|:---|:---|:---|
| **What is combined?** | Features from different modalities | Predictions from separate modality models |
| **When are they combined?** | Before the prediction model processes the inputs | After each modality model makes a prediction |
| **Typical method** | Join feature vectors and feed them to one model | Average probabilities, vote on labels, or learn how to combine predictions |
| **Main advantage** | The model can learn from patterns across modalities if its architecture supports them | Each modality can have its own preprocessing and model |
| **Main limitation** | Inputs must be aligned, and combining many features can increase the risk of overfitting | The fusion step has access to model outputs only, so it can lose useful feature detail |
| **Missing modality** | Requires a planned approach, such as imputation or a model designed for missing inputs | Can use the available models under a predefined rule |
| **Good fit when** | Modalities are usually available together and their joint patterns matter | Modalities need different models or are sometimes unavailable |

### Real Use Case

Run these cells before presenting or working through the example. They download the [BIDMC files](https://physionet.org/content/bidmc/1.0.0/), build 30-second windows and split by patient. The initial download is about 153 MB. Existing files are reused. The notebook has all the code and needs no helper file.

Age is the sole tabular predictor. We encode 90+ as 90 for this demonstration and omit records without a usable age. Each positive label comes from a median valid monitor respiratory rate of at least 20 breaths/min. Respiratory rate is used only for the label.

In [1]:
# Install once if needed, then restart the kernel.
# %pip install numpy pandas matplotlib scikit-learn

from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [3]:
base_url = "https://physionet.org/files/bidmc/1.0.0/bidmc_csv/"
data_dir = Path("data/bidmc")
data_dir.mkdir(parents=True, exist_ok=True)
record_numbers = range(1, 54)

for number in record_numbers:
    for ending in ["Signals.csv", "Numerics.csv", "Fix.txt"]:
        filename = f"bidmc_{number:02d}_{ending}"
        path = data_dir / filename

        if not path.exists():
            temporary = path.with_suffix(path.suffix + ".part")
            urlretrieve(base_url + filename, temporary)
            temporary.replace(path)

print("Original files ready:", len(list(data_dir.glob("bidmc_*"))))

In [ ]:
rows = []

for number in record_numbers:
    prefix = data_dir / f"bidmc_{number:02d}"
    metadata = Path(str(prefix) + "_Fix.txt").read_text()
    age = None
    patient = None

    for line in metadata.splitlines():
        if line.startswith("Age:"):
            value = line.split(":", 1)[1].strip().rstrip("+")
            if value.isdigit():
                age = int(value)
        if line.startswith("MIMIC II matched wdb ID:"):
            patient = line.split(":", 1)[1].strip()

    if age is None or not patient:
        continue

    signals = pd.read_csv(str(prefix) + "_Signals.csv", nrows=30000)
    monitor = pd.read_csv(str(prefix) + "_Numerics.csv", nrows=240)
    signals.columns = signals.columns.str.strip()
    monitor.columns = monitor.columns.str.strip()

    for start in range(0, 240, 30):
        end = start + 30
        wave_time = signals["Time [s]"]
        monitor_time = monitor["Time [s]"]
        ecg = signals.loc[(wave_time >= start) & (wave_time < end), "II"].to_numpy()
        rr = monitor.loc[(monitor_time >= start) & (monitor_time < end), "RESP"].to_numpy()
        rr = rr[np.isfinite(rr) & (rr > 0)]

        if len(ecg) != 3750 or not np.isfinite(ecg).all() or len(rr) < 24:
            continue

        rows.append({
            "patient": patient,
            "record": number,
            "start_s": start,
            "ecg_spread": np.std(ecg),
            "ecg_change": np.abs(np.diff(ecg)).mean(),
            "age": age,
            "higher_rr": int(np.median(rr) >= 20)
        })

data = pd.DataFrame(rows)
split = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_rows, test_rows = next(split.split(data, groups=data["patient"]))
train = data.iloc[train_rows]
test = data.iloc[test_rows]
y_train = train["higher_rr"]
y_test = test["higher_rr"]
assert not set(train["patient"]) & set(test["patient"])
assert y_train.nunique() == y_test.nunique() == 2

def predict_risk(columns):
    model = make_pipeline(StandardScaler(), LogisticRegression())
    model.fit(train[columns], y_train)
    return model.predict_proba(test[columns])[:, 1]

def scores(probability):
    prediction = probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    return [tp / (tp + fn), tn / (tn + fp), (tp + tn) / len(y_test)]


## Step 1 - Look at the input

One row represents 30 seconds of ECG. The two ECG features measure signal spread and mean absolute change between samples. Age comes from the recording metadata.

The target is 1 if the window’s median monitor respiratory rate is at least 20 breaths per minute. The split keeps patients separate.

In [ ]:
ecg_columns = ["ecg_spread", "ecg_change"]
age_columns = ["age"]

display(train[ecg_columns + age_columns + ["higher_rr"]].head(4))
print("Training windows:", len(train), "| Test windows:", len(test))

## Step 2 - Fit the single-input models

Use the supplied function to fit one logistic regression per input. The function learns scaling from the training data and returns the probability of class 1 for each test window.

In [ ]:
p_ecg = predict_risk(ecg_columns)
p_age = predict_risk(age_columns)

## Step 3 - Early fusion

Join the feature names. The same function now fits one model using all three columns together.

In [ ]:
joint_columns = ecg_columns + age_columns
p_early = predict_risk(joint_columns)

## Step 4 - Late fusion

The two single-input models already made predictions for the same test rows in the same order. Average the matching probabilities.

In [ ]:
p_late = (p_ecg + p_age) / 2

## Step 5 - Compare on held-out patients

The supplied scoring function applies a threshold of 0.50 and calculates the three metrics from the confusion matrix. Values below are percentages.

**Read the result:** Early and late fusion have the same accuracy. Early fusion catches more positives, while late fusion raises fewer false alarms.

In [ ]:
results = pd.DataFrame(
    [scores(p_ecg), scores(p_age), scores(p_early), scores(p_late)],
    index=["ECG only", "Age only", "Early fusion", "Late fusion"],
    columns=["Sensitivity", "Specificity", "Accuracy"]
)
display((100 * results).round(1))

## Exercise 1 · Trace one prediction (10 min)

Pick a test row. Round and compare the ECG, age and late-fusion probabilities. Verify the average by hand. Does the combined decision match the ECG-only decision at 0.50?

**Starter:** Set the row position below and use it to index the probability arrays.

In [ ]:
row = 0

# TODO: print p_ecg[row], p_age[row] and p_late[row].
# TODO: compare the ECG-only and late-fusion decisions.

## Exercise 2 · Does age help here? (10 min)

Compare the ECG-only, early-fusion and late-fusion rows in the results table. Which model detects more positive windows? Which raises more false alarms?

Then suppose age is missing. Which already fitted model could still make a prediction? How would you check its performance on windows with missing age?